## Data Loading & Profiling

In [ ]:
import pandas as pd
import os

DATA_PATH = "../data/raw"
PROCESSED_PATH = "../data/processed"

patients = pd.read_csv(os.path.join(DATA_PATH, "patients.csv"))
admissions = pd.read_csv(os.path.join(DATA_PATH, "admissions.csv"))
doctors = pd.read_csv(os.path.join(DATA_PATH, "doctors.csv"))
vitals = pd.read_csv(os.path.join(DATA_PATH, "vitals.csv"))
treatments = pd.read_csv(os.path.join(DATA_PATH, "treatments.csv"))

datasets = {
    "Patients": patients,
    "Admissions": admissions,
    "Doctors": doctors,
    "Vitals": vitals,
    "Treatments": treatments
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")


In [ ]:
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name)
    print('=' * 60)
    print(df.info())


In [ ]:
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name)
    print('=' * 60)
    print(df.isnull().sum())


In [ ]:
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name)
    print('=' * 60)
    print(f"Duplicate rows: {df.duplicated().sum()}")

    id_column = {
        "Patients": "patient_id",
        "Admissions": "admission_id",
        "Doctors": "doctor_id",
        "Vitals": "vital_id",
        "Treatments": "treatment_id"
    }[name]

    print(f"Duplicate {id_column}: {df[id_column].duplicated().sum()}")


## Data Quality Validation

In [ ]:
date_columns = {
    "Patients": ["dob"],
    "Admissions": ["admission_date", "discharge_date"],
    "Vitals": ["recorded_time"],
    "Treatments": ["treatment_date"]
}

for name, columns in date_columns.items():
    print(f"\n{'=' * 60}")
    print(name)
    print('=' * 60)

    for column in columns:
        converted = pd.to_datetime(datasets[name][column], errors="coerce")
        print(f"{column}:")
        print(f"  Invalid dates: {converted.isna().sum()}")
        print(f"  Min: {converted.min()}")
        print(f"  Max: {converted.max()}")


In [ ]:
admission_dates = admissions[["admission_id", "admission_date", "discharge_date"]].copy()

admission_dates["admission_date"] = pd.to_datetime(admission_dates["admission_date"])
admission_dates["discharge_date"] = pd.to_datetime(admission_dates["discharge_date"])

vitals_check = vitals.merge(admission_dates, on="admission_id", how="left")
vitals_check["recorded_time"] = pd.to_datetime(vitals_check["recorded_time"])

vitals_check["within_admission"] = (
    (vitals_check["recorded_time"] >= vitals_check["admission_date"]) &
    (vitals_check["recorded_time"] <= vitals_check["discharge_date"])
)

treatments_check = treatments.merge(admission_dates, on="admission_id", how="left")
treatments_check["treatment_date"] = pd.to_datetime(treatments_check["treatment_date"])

treatments_check["within_admission"] = (
    (treatments_check["treatment_date"] >= treatments_check["admission_date"]) &
    (treatments_check["treatment_date"] <= treatments_check["discharge_date"])
)

print("Vitals within admission period:")
print(vitals_check["within_admission"].value_counts())

print("\nTreatments within admission period:")
print(treatments_check["within_admission"].value_counts())


In [ ]:
print("Admissions → Patients:", (~admissions["patient_id"].isin(patients["patient_id"])).sum())
print("Admissions → Doctors:", (~admissions["doctor_id"].isin(doctors["doctor_id"])).sum())
print("Vitals → Admissions:", (~vitals["admission_id"].isin(admissions["admission_id"])).sum())
print("Treatments → Admissions:", (~treatments["admission_id"].isin(admissions["admission_id"])).sum())


In [ ]:
admissions_check = admissions.copy()

admissions_check["admission_date"] = pd.to_datetime(admissions_check["admission_date"])
admissions_check["discharge_date"] = pd.to_datetime(admissions_check["discharge_date"])

admissions_check["length_of_stay"] = (
    admissions_check["discharge_date"] - admissions_check["admission_date"]
).dt.days

print("Admission date after discharge date:",
      (admissions_check["admission_date"] > admissions_check["discharge_date"]).sum())

print("Same-day admissions:",
      (admissions_check["length_of_stay"] == 0).sum())

print("Negative length of stay:",
      (admissions_check["length_of_stay"] < 0).sum())

print("\nLength of stay summary:")
print(admissions_check["length_of_stay"].describe())


In [ ]:
admissions_sorted = admissions_check.sort_values(
    ["patient_id", "admission_date", "discharge_date"]
).copy()

admissions_sorted["previous_discharge"] = (
    admissions_sorted.groupby("patient_id")["discharge_date"].cummax()
    .groupby(admissions_sorted["patient_id"]).shift(1)
)

overlapping_admissions = admissions_sorted[
    admissions_sorted["admission_date"] < admissions_sorted["previous_discharge"]
]

print("Total overlapping admissions:", len(overlapping_admissions))
print("Patients affected:", overlapping_admissions["patient_id"].nunique())

print("\nSample overlapping admissions:")
print(
    overlapping_admissions[
        ["patient_id", "admission_id", "admission_date",
         "discharge_date", "previous_discharge"]
    ].head(10).to_string(index=False)
)


In [ ]:
blood_pressure_parts = vitals["blood_pressure"].astype(str).str.split("/", expand=True)

print("Blood pressure formats:")
print(vitals["blood_pressure"].value_counts().head(20))

print("\nRecords without exactly two blood pressure components:",
      (blood_pressure_parts.shape[1] != 2))

print("\nInvalid systolic values:",
      pd.to_numeric(blood_pressure_parts[0], errors="coerce").isna().sum())

print("Invalid diastolic values:",
      pd.to_numeric(blood_pressure_parts[1], errors="coerce").isna().sum())

print("\nSystolic range:")
print(pd.to_numeric(blood_pressure_parts[0], errors="coerce").describe())

print("\nDiastolic range:")
print(pd.to_numeric(blood_pressure_parts[1], errors="coerce").describe())


In [ ]:
missing_conditions = patients[patients["chronic_conditions"].isna()]

print("Patients with missing chronic conditions:", len(missing_conditions))

print("\nGender distribution:")
print(missing_conditions["gender"].value_counts())

print("\nSample records:")
print(missing_conditions.head(10).to_string(index=False))


In [ ]:
patients_check = patients.copy()

patients_check["dob"] = pd.to_datetime(patients_check["dob"])
reference_date = pd.Timestamp("2026-03-20")

patients_check["age"] = (
    reference_date.year
    - patients_check["dob"].dt.year
    - (
        (patients_check["dob"].dt.month > reference_date.month)
        | (
            (patients_check["dob"].dt.month == reference_date.month)
            & (patients_check["dob"].dt.day > reference_date.day)
        )
    ).astype(int)
)

print("Age summary:")
print(patients_check["age"].describe())

print("\nPatients younger than 0:",
      (patients_check["age"] < 0).sum())

print("Patients older than 100:",
      (patients_check["age"] > 100).sum())


In [ ]:
diagnosis_values = admissions["diagnosis"].dropna().astype(str).str.strip()

print("Missing diagnoses:", admissions["diagnosis"].isna().sum())
print("Unique diagnoses:", diagnosis_values.nunique())

print("\nDiagnosis values:")
print(diagnosis_values.value_counts())

print("\nBlank diagnosis values:",
      (diagnosis_values == "").sum())


In [ ]:
doctor_contacts = doctors["contact_no"].astype(str)

print("Duplicate doctor IDs:", doctors["doctor_id"].duplicated().sum())
print("Missing specializations:", doctors["specialization"].isna().sum())
print("Unique specializations:", doctors["specialization"].nunique())

print("\nDoctor specializations:")
print(doctors["specialization"].value_counts())

print("\nContact numbers with fewer than 10 digits:",
      (doctor_contacts.str.len() < 10).sum())

print("Contact numbers with more than 10 digits:",
      (doctor_contacts.str.len() > 10).sum())


In [ ]:
print("Heart rate below 60:", (vitals["heart_rate"] < 60).sum())
print("Heart rate above 100:", (vitals["heart_rate"] > 100).sum())

print("Oxygen level below 95:", (vitals["oxygen_level"] < 95).sum())

print("Temperature below 98.6:", (vitals["temperature"] < 98.6).sum())
print("Temperature above 100.4:", (vitals["temperature"] > 100.4).sum())


## Feature Engineering

In [ ]:
valid_vitals = vitals_check[
    vitals_check["within_admission"]
].copy()

valid_treatments = treatments_check[
    treatments_check["within_admission"]
].copy()

print("Valid vitals:", len(valid_vitals))
print("Valid treatments:", len(valid_treatments))

print("\nValid vitals by admission:")
print(valid_vitals["admission_id"].nunique())

print("Valid treatments by admission:")
print(valid_treatments["admission_id"].nunique())


In [ ]:
blood_pressure_parts = valid_vitals["blood_pressure"].str.split("/", expand=True)

valid_vitals["systolic_bp"] = pd.to_numeric(blood_pressure_parts[0])
valid_vitals["diastolic_bp"] = pd.to_numeric(blood_pressure_parts[1])

print("Systolic BP:")
print(valid_vitals["systolic_bp"].describe())

print("\nDiastolic BP:")
print(valid_vitals["diastolic_bp"].describe())

print("\nMissing systolic BP:", valid_vitals["systolic_bp"].isna().sum())
print("Missing diastolic BP:", valid_vitals["diastolic_bp"].isna().sum())


In [ ]:
patients_clean = patients_check.copy()

patients_clean["chronic_conditions"] = patients_clean["chronic_conditions"].fillna("")

patients_clean["chronic_condition_count"] = (
    patients_clean["chronic_conditions"]
    .str.split(",")
    .apply(lambda x: len([condition for condition in x if condition.strip()]))
)

patients_clean["contact_no"] = patients_clean["contact_no"].astype(str)

print("Chronic condition count:")
print(patients_clean["chronic_condition_count"].value_counts().sort_index())

print("\nMissing chronic condition values after treatment:",
      patients_clean["chronic_conditions"].isna().sum())


In [ ]:
admission_features = admissions_history.copy()

admission_features = admission_features.merge(
    patients_clean[
        ["patient_id", "gender", "age", "chronic_condition_count"]
    ],
    on="patient_id",
    how="left"
)

admission_features = admission_features.merge(
    doctors[
        ["doctor_id", "specialization"]
    ],
    on="doctor_id",
    how="left"
)

print("Rows:", len(admission_features))
print("Columns:", len(admission_features.columns))

print("\nMissing values:")
print(admission_features[
    ["gender", "age", "chronic_condition_count", "specialization"]
].isna().sum())


In [ ]:
vital_features = (
    valid_vitals
    .groupby("admission_id")
    .agg(
        avg_heart_rate=("heart_rate", "mean"),
        min_oxygen_level=("oxygen_level", "min"),
        avg_temperature=("temperature", "mean"),
        avg_systolic_bp=("systolic_bp", "mean"),
        avg_diastolic_bp=("diastolic_bp", "mean"),
        abnormal_heart_rate_count=("heart_rate", lambda x: (x > 100).sum()),
        abnormal_oxygen_count=("oxygen_level", lambda x: (x < 95).sum()),
        abnormal_temperature_count=(
            "temperature",
            lambda x: ((x < 98.6) | (x > 100.4)).sum()
        ),
        vital_record_count=("vital_id", "count")
    )
    .reset_index()
)

vital_features["total_abnormal_vitals"] = (
    vital_features["abnormal_heart_rate_count"]
    + vital_features["abnormal_oxygen_count"]
    + vital_features["abnormal_temperature_count"]
)

print("Admissions with vital features:", len(vital_features))
print("\nVital feature sample:")
print(vital_features.head().to_string(index=False))


In [ ]:
treatment_features = (
    valid_treatments
    .groupby("admission_id")
    .agg(
        treatment_count=("treatment_id", "count"),
        unique_procedures=("procedure", "nunique"),
        unique_medications=("medication", "nunique"),
        unique_dosages=("dosage", "nunique")
    )
    .reset_index()
)

print("Admissions with treatment features:", len(treatment_features))

print("\nTreatment feature sample:")
print(treatment_features.head().to_string(index=False))


In [ ]:
modeling_data = admission_features.merge(
    vital_features,
    on="admission_id",
    how="left"
)

modeling_data = modeling_data.merge(
    treatment_features,
    on="admission_id",
    how="left"
)

modeling_data["readmitted_30d"] = modeling_data["readmitted_30d"].astype(int)

print("Final modeling rows:", len(modeling_data))
print("Final modeling columns:", len(modeling_data.columns))

print("\nTarget distribution:")
print(modeling_data["readmitted_30d"].value_counts())

print("\nMissing values in engineered features:")
print(
    modeling_data[
        [
            "avg_heart_rate",
            "min_oxygen_level",
            "avg_temperature",
            "avg_systolic_bp",
            "avg_diastolic_bp",
            "treatment_count"
        ]
    ].isna().sum()
)


In [ ]:
print("Duplicate admission IDs:",
      modeling_data["admission_id"].duplicated().sum())

print("Duplicate rows:",
      modeling_data.duplicated().sum())

print("Missing admission IDs:",
      modeling_data["admission_id"].isna().sum())

print("Invalid target values:",
      (~modeling_data["readmitted_30d"].isin([0, 1])).sum())

print("Total admissions:",
      modeling_data["admission_id"].nunique())

print("\nFinal dataset shape:")
print(modeling_data.shape)


## 30-Day Readmission Target

In [ ]:
admissions_target = admissions.copy()

admissions_target["admission_date"] = pd.to_datetime(
    admissions_target["admission_date"]
)
admissions_target["discharge_date"] = pd.to_datetime(
    admissions_target["discharge_date"]
)

current_admissions = admissions_target[
    ["admission_id", "patient_id", "discharge_date"]
].copy()

future_admissions = admissions_target[
    ["admission_id", "patient_id", "admission_date"]
].copy()

current_admissions = current_admissions.rename(
    columns={"admission_id": "current_admission_id"}
)

future_admissions = future_admissions.rename(
    columns={
        "admission_id": "future_admission_id",
        "admission_date": "future_admission_date"
    }
)

readmission_pairs = current_admissions.merge(
    future_admissions,
    on="patient_id",
    how="left"
)

readmission_pairs = readmission_pairs[
    readmission_pairs["future_admission_id"]
    != readmission_pairs["current_admission_id"]
]

readmission_pairs["days_to_next_admission"] = (
    readmission_pairs["future_admission_date"]
    - readmission_pairs["discharge_date"]
).dt.days

readmission_pairs = readmission_pairs[
    readmission_pairs["days_to_next_admission"] >= 0
]

next_readmission = (
    readmission_pairs
    .groupby("current_admission_id")["days_to_next_admission"]
    .min()
    .reset_index()
)

admissions_target = admissions_target.merge(
    next_readmission,
    left_on="admission_id",
    right_on="current_admission_id",
    how="left"
)

admissions_target["readmitted_30d"] = (
    admissions_target["days_to_next_admission"]
    .between(0, 30, inclusive="both")
).astype(int)

print("Corrected target distribution:")
print(admissions_target["readmitted_30d"].value_counts())

print("\nReadmission percentage:")
print(
    round(
        admissions_target["readmitted_30d"].mean() * 100,
        2
    ),
    "%"
)

print("\nSample readmission cases:")
print(
    admissions_target[
        admissions_target["readmitted_30d"] == 1
    ][
        [
            "admission_id",
            "patient_id",
            "discharge_date",
            "days_to_next_admission",
            "readmitted_30d"
        ]
    ].head(10)
)


In [ ]:
max_discharge_date = admissions_target["discharge_date"].max()
observation_cutoff = max_discharge_date - pd.Timedelta(days=30)

complete_followup = admissions_target[
    admissions_target["discharge_date"] <= observation_cutoff
].copy()

incomplete_followup = admissions_target[
    admissions_target["discharge_date"] > observation_cutoff
].copy()

print("Maximum discharge date:", max_discharge_date.date())
print("30-day observation cutoff:", observation_cutoff.date())

print("\nAdmissions with complete 30-day follow-up:", len(complete_followup))
print("Admissions without complete 30-day follow-up:", len(incomplete_followup))

print("\nTarget distribution after applying the observation window:")
print(complete_followup["readmitted_30d"].value_counts())

print("\nReadmission percentage:")
print(
    round(
        complete_followup["readmitted_30d"].mean() * 100,
        2
    ),
    "%"
)


In [ ]:
modeling_data_clean = modeling_data[
    modeling_data["admission_id"].isin(
        complete_followup["admission_id"]
    )
].copy()

modeling_data_clean = modeling_data_clean.drop(
    columns=["readmitted_30d", "days_to_next_admission"],
    errors="ignore"
)

modeling_data_clean = modeling_data_clean.merge(
    complete_followup[
        ["admission_id", "readmitted_30d"]
    ],
    on="admission_id",
    how="inner"
)

print("Clean modeling dataset shape:", modeling_data_clean.shape)

print("\nTarget distribution:")
print(modeling_data_clean["readmitted_30d"].value_counts())

print("\nTarget percentage:")
print(
    round(
        modeling_data_clean["readmitted_30d"].mean() * 100,
        2
    ),
    "%"
)


## Machine Learning Preparation

In [ ]:
ml_features = [
    "diagnosis",
    "gender",
    "age",
    "chronic_condition_count",
    "specialization",
    "length_of_stay",
    "avg_heart_rate",
    "min_oxygen_level",
    "avg_temperature",
    "avg_systolic_bp",
    "avg_diastolic_bp",
    "abnormal_heart_rate_count",
    "abnormal_oxygen_count",
    "abnormal_temperature_count",
    "vital_record_count",
    "total_abnormal_vitals",
    "treatment_count",
    "unique_procedures",
    "unique_medications",
    "unique_dosages"
]

X_clean = modeling_data_clean[ml_features].copy()
y_clean = modeling_data_clean["readmitted_30d"].copy()

categorical_features = X_clean.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_features = X_clean.select_dtypes(
    exclude=["object", "string"]
).columns.tolist()

print("Feature matrix shape:", X_clean.shape)
print("Target shape:", y_clean.shape)

print("\nCategorical features:")
print(categorical_features)

print("\nNumeric features:")
print(numeric_features)

print("\nTarget distribution:")
print(y_clean.value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_clean,
    y_clean,
    test_size=0.20,
    stratify=y_clean,
    random_state=42
)

print("Training set:", X_train_raw.shape)
print("Testing set:", X_test_raw.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

print("\nTraining readmission rate:")
print(round(y_train.mean() * 100, 2), "%")

print("Testing readmission rate:")
print(round(y_test.mean() * 100, 2), "%")


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train_raw)
X_test_processed = preprocessor.transform(X_test_raw)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

print("\nPreprocessing completed successfully.")


## Model Training & Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

logistic_model.fit(X_train_processed, y_train)

print("Logistic Regression model trained successfully.")
print("Number of iterations:", logistic_model.n_iter_[0])


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

logistic_pred = logistic_model.predict(X_test_processed)
logistic_prob = logistic_model.predict_proba(X_test_processed)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, logistic_pred), 4))
print("Precision:", round(precision_score(y_test, logistic_pred), 4))
print("Recall:", round(recall_score(y_test, logistic_pred), 4))
print("F1 Score:", round(f1_score(y_test, logistic_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, logistic_prob), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, logistic_pred))

print("\nClassification Report:")
print(classification_report(y_test, logistic_pred))


In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train_processed, y_train)

print("Random Forest model trained successfully.")
print("Number of trees:", random_forest_model.n_estimators)


In [ ]:
rf_pred = random_forest_model.predict(X_test_processed)
rf_prob = random_forest_model.predict_proba(X_test_processed)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("Precision:", round(precision_score(y_test, rf_pred), 4))
print("Recall:", round(recall_score(y_test, rf_pred), 4))
print("F1 Score:", round(f1_score(y_test, rf_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_prob), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))


In [ ]:
from xgboost import XGBClassifier

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_processed, y_train)

print("XGBoost model trained successfully.")
print("Negative class:", negative_count)
print("Positive class:", positive_count)
print("Scale positive weight:", round(scale_pos_weight, 4))
print("Number of trees:", xgb_model.n_estimators)


In [ ]:
xgb_pred = xgb_model.predict(X_test_processed)
xgb_prob = xgb_model.predict_proba(X_test_processed)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, xgb_pred), 4))
print("Precision:", round(precision_score(y_test, xgb_pred), 4))
print("Recall:", round(recall_score(y_test, xgb_pred), 4))
print("F1 Score:", round(f1_score(y_test, xgb_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, xgb_prob), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))

print("\nClassification Report:")
print(classification_report(y_test, xgb_pred))


## Model Comparison & Feature Importance

In [ ]:
feature_names = preprocessor.get_feature_names_out()

xgb_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_model.feature_importances_
})

xgb_importance = xgb_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print("Top 15 XGBoost features:")
print(xgb_importance.head(15).to_string(index=False))


In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        accuracy_score(y_test, logistic_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred)
    ],
    "Precision": [
        precision_score(y_test, logistic_pred),
        precision_score(y_test, rf_pred),
        precision_score(y_test, xgb_pred)
    ],
    "Recall": [
        recall_score(y_test, logistic_pred),
        recall_score(y_test, rf_pred),
        recall_score(y_test, xgb_pred)
    ],
    "F1 Score": [
        f1_score(y_test, logistic_pred),
        f1_score(y_test, rf_pred),
        f1_score(y_test, xgb_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, logistic_prob),
        roc_auc_score(y_test, rf_prob),
        roc_auc_score(y_test, xgb_prob)
    ]
})

model_comparison.iloc[:, 1:] = model_comparison.iloc[:, 1:].round(4)

print(model_comparison.to_string(index=False))


In [ ]:
modeling_data_clean.to_csv(
    os.path.join(PROCESSED_PATH, "admission_modeling_data_clean.csv"),
    index=False
)

model_comparison.to_csv(
    os.path.join(PROCESSED_PATH, "model_comparison.csv"),
    index=False
)

print("Saved files:")
print("1. admission_modeling_data_clean.csv")
print("2. model_comparison.csv")

print("\nClean modeling rows:", len(modeling_data_clean))
print("Model comparison rows:", len(model_comparison))


## Risk Score Generation

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_logistic_model = CalibratedClassifierCV(
    logistic_model,
    method="sigmoid",
    cv=5
)

calibrated_logistic_model.fit(
    X_train_processed,
    y_train
)

calibrated_prob = calibrated_logistic_model.predict_proba(
    X_test_processed
)[:, 1]

calibrated_pred = (
    calibrated_prob >= 0.50
).astype(int)

print("Calibrated Logistic Regression trained successfully.")

print("\nCalibrated model performance:")
print("Accuracy:", round(accuracy_score(y_test, calibrated_pred), 4))
print("Precision:", round(precision_score(y_test, calibrated_pred), 4))
print("Recall:", round(recall_score(y_test, calibrated_pred), 4))
print("F1 Score:", round(f1_score(y_test, calibrated_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, calibrated_prob), 4))

print("\nAverage predicted probability:")
print(round(calibrated_prob.mean() * 100, 2), "%")


In [ ]:
X_clean_processed = preprocessor.transform(X_clean)

final_predictions = modeling_data_clean[
    [
        "admission_id",
        "patient_id",
        "admission_date",
        "discharge_date",
        "diagnosis",
        "doctor_id",
        "room_no",
        "length_of_stay",
        "gender",
        "age",
        "chronic_condition_count",
        "specialization",
        "readmitted_30d"
    ]
].copy()

final_predictions["readmission_probability"] = (
    calibrated_logistic_model.predict_proba(
        X_clean_processed
    )[:, 1]
)

final_predictions.to_csv(
    os.path.join(
        PROCESSED_PATH,
        "readmission_risk_scores.csv"
    ),
    index=False
)

print("Readmission risk scores saved successfully.")

print("\nRows saved:", len(final_predictions))

print("\nAverage predicted probability:")
print(
    round(
        final_predictions["readmission_probability"].mean() * 100,
        2
    ),
    "%"
)

print("\nHighest predicted probabilities:")
print(
    final_predictions[
        [
            "admission_id",
            "patient_id",
            "readmission_probability"
        ]
    ]
    .sort_values(
        "readmission_probability",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)


## Power BI Risk Export

In [ ]:
import pandas as pd

risk_scores = pd.read_csv("../data/processed/readmission_risk_scores.csv")

risk_scores.head()


In [ ]:
risk_scores["readmission_probability"].describe()


In [ ]:
q33 = risk_scores["readmission_probability"].quantile(0.33)
q67 = risk_scores["readmission_probability"].quantile(0.67)

risk_levels = pd.cut(
    risk_scores["readmission_probability"],
    bins=[-float("inf"), q33, q67, float("inf")],
    labels=["Low", "Medium", "High"]
)

print("33rd percentile:", q33)
print("67th percentile:", q67)
print()
print(risk_levels.value_counts().sort_index())


In [ ]:
risk_export = pd.DataFrame({
    "risk_id": range(9001, 9001 + len(risk_scores)),
    "admission_id": risk_scores["admission_id"],
    "prediction_date": pd.to_datetime(risk_scores["discharge_date"]).dt.date,
    "risk_score": risk_scores["readmission_probability"].round(2),
    "risk_level": pd.cut(
        risk_scores["readmission_probability"],
        bins=[-float("inf"), q33, q67, float("inf")],
        labels=["Low", "Medium", "High"]
    )
})

risk_export.to_csv("../data/processed/readmission_risk.csv", index=False)

print(risk_export.head())
print()
print(risk_export.shape)
print()
print(risk_export["risk_level"].value_counts().sort_index())
